In [86]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_classic.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA


In [87]:
# Read the PDF from the folder

loader=PyPDFDirectoryLoader("./us_census")
documents=loader.load()
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
final_document=text_splitter.split_documents(documents)
final_document[0]

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/VXSVPH+Gotham-Book', '/Encoding': IndirectObject(137, 0, 2292079507408), '/FirstChar': 1, '/FontDescriptor': IndirectObject(200, 0, 2292079507408), '/LastChar': 173, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [861, 664, 616, 631, 591, 571, 782, 430, 434, 664, 664, 556, 404, 498, 646, 245, 616, 579, 263, 587, 742, 434, 300, 600, 300, 592, 405, 616, 664, 263, 656, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 599, 868, 715, 574, 948, 367, 790, 641, 408, 625, 508, 600, 625, 759, 358, 722, 647, 244, 850, 760, 750, 723, 291, 716, 569, 668, 719, 1107, 553, 615, 790, 784, 600, 254, 600, 600, 600, 600, 670, 0, 0, 0, 0, 0, 0, 0, 640]}, but is not in

Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.2 (Windows)', 'creationdate': '2023-09-09T07:52:17-04:00', 'author': 'U.S. Census Bureau', 'keywords': 'acsbr-015', 'moddate': '2023-09-12T14:44:47+01:00', 'title': 'Health Insurance Coverage Status and Type by Geography: 2021 and 2022', 'trapped': '/false', 'source': 'us_census\\acsbr-015.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1'}, page_content='Health Insurance Coverage Status and Type \nby Geography: 2021 and 2022\nAmerican Community Survey Briefs\nACSBR-015\nIssued September 2023\nDouglas Conway and Breauna Branch\nINTRODUCTION\nDemographic shifts as well as economic and govern-\nment policy changes can affect people’s access to \nhealth coverage. For example, between 2021 and 2022, \nthe labor market continued to improve, which may \nhave affected private coverage in the United States \nduring that time.1 Public policy changes included \nthe renewal of the Public Health Emergency, wh

In [88]:
len(final_document)

316

In [89]:
# Embedding using HuggingFace

huggingface_embedding=HuggingFaceBgeEmbeddings(
    model_name='BAAI/bge-small-en-v1.5', # sentence-transformers/all-MiniLM-16-v2
    model_kwargs={'device':'cpu'},
    encode_kwargs={'normalize_embeddings':True},
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3491.07it/s]


In [90]:
import numpy as np
np.array(huggingface_embedding.embed_query(final_document[0].page_content))

array([-0.07903482, -0.01134115, -0.02312098,  0.02844463,  0.05053343,
        0.05317824, -0.0190779 ,  0.03456027, -0.10211368, -0.02915701,
        0.08524263,  0.05650725, -0.02545437, -0.03308488, -0.00635736,
        0.04090868, -0.00628107,  0.00356743, -0.03854126,  0.03667683,
       -0.042898  ,  0.03425248, -0.03116902, -0.0379373 ,  0.0172839 ,
        0.01214927,  0.00653118,  0.01463563, -0.05529055, -0.15320711,
        0.00730849,  0.03202942, -0.04701127, -0.01595974,  0.01874447,
        0.02642939, -0.02306377,  0.0843804 ,  0.04182489,  0.05278176,
       -0.03057602,  0.0156426 , -0.01689074,  0.00529407, -0.02417436,
        0.00412995, -0.01889937, -0.00150626, -0.0083694 , -0.03390066,
        0.03515956, -0.00553131,  0.04910937,  0.05971859,  0.05615962,
       -0.05105156,  0.01475136, -0.01849963, -0.03284641,  0.03576629,
        0.04947706, -0.00938883, -0.26202118,  0.09750332,  0.0171569 ,
        0.04781385, -0.00556319, -0.00298308, -0.02207356, -0.04

In [91]:
np.array(huggingface_embedding.embed_query(final_document[0].page_content)).shape

(384,)

In [92]:
# VectorStore creation
vectorstores=FAISS.from_documents(final_document[:120],huggingface_embedding)

In [93]:
# Query using similarity search
query="WHAT IS HEALTH INSURANCE COVERAGE?"
relevant_document=vectorstores.similarity_search(query)
print(relevant_document[0].page_content)

2 U.S. Census Bureau
WHAT IS HEALTH INSURANCE COVERAGE?
This brief presents state-level estimates of health insurance coverage 
using data from the American Community Survey (ACS). The  
U.S. Census Bureau conducts the ACS throughout the year; the 
survey asks respondents to report their coverage at the time of 
interview. The resulting measure of health insurance coverage, 
therefore, reflects an annual average of current comprehensive 
health insurance coverage status.* This uninsured rate measures a 
different concept than the measure based on the Current Population 
Survey Annual Social and Economic Supplement (CPS ASEC). 
For reporting purposes, the ACS broadly classifies health insurance 
coverage as private insurance or public insurance. The ACS defines 
private health insurance as a plan provided through an employer 
or a union, coverage purchased directly by an individual from an 
insurance company or through an exchange (such as healthcare.


In [94]:
retriever=vectorstores.as_retriever(search_type="similarity",serach_kwargs={"k":3})
print(retriever)

tags=['FAISS', 'HuggingFaceBgeEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002159BBE8230> search_kwargs={}


In [135]:
from huggingface_hub import InferenceClient
import os
from dotenv import load_dotenv
load_dotenv()

token = os.getenv("HUGGINGFACEHUB_API_TOKEN")
client = InferenceClient(
    api_key=token
)
hf1=client.chat.completions.create(
    model="openai/gpt-oss-120b",
    max_tokens=300,
    temperature=0.1,
    messages=[
        {
            "role": "user",
            "content": "HOW POVERTY IS MEASURED?"
        }
    ],
)
print(hf1.choices[0].message.content)


**Health insurance coverage** is the portion of medical expenses that a health‑insurance policy agrees to pay on your behalf. In other words, it’s the “shield” that protects you (or your family) from the full cost of health‑care services—doctor visits, hospital stays, prescription drugs, preventive care, and many other treatments.

Below is a concise breakdown of what health‑insurance coverage means, what it typically includes, and how it works in practice.

---

## 1. The Core Idea

| Term | What It Means |
|------|---------------|
| **Coverage** | The set of health‑care services that the insurer will pay for (either fully or partially) according to the plan’s rules. |
| **Benefit** | The actual dollar amount or percentage the insurer pays for a covered service (e.g., “80% of the allowed amount”). |
| **Cost‑Sharing** | The portion you’re still responsible for (deductibles, copayments, coinsurance, out‑of‑pocket maximums). |

**Bottom line:** Coverage tells you *what* is paid, while c

In [136]:
prompt_template="""
Use the following piece of context to answer the question asked.
Please try to provide the answer only based on the context

{context}
Question:{question}

Helpful Answers:
 """

In [137]:
prompt=PromptTemplate(template=prompt_template,input_variables=["context","question"])

In [144]:
from langchain_classic.chains import RetrievalQA

retrievalQA = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={
        "prompt": prompt
    }
)

print("RetrievalQA created successfully")

RetrievalQA created successfully


In [145]:
query="""
DIFFERENCES IN THE
UNINSURED RATE BY STATE
IN 2022
"""

In [146]:
# Call the QA chain with our query.
result = retrievalQA.invoke({"query": query})
print(result['result'])

In 2022 the share of people who were uninsured varied widely from state to state. The lowest uninsured rate was **2.4 % in Massachusetts**, while the highest was **16.6 % in Texas**. Both figures are measured against the national uninsured rate of **8.0 %**.  

Because of this spread, **ten of the fifteen states whose
